In [ ]:
import pandas as pd
from sqlalchemy import create_engine

### Подключение к MS SQL

In [ ]:
server = 'localhost'
database = 'ChudoObuv'

# Строка подключения
connection_string = f"mssql+pyodbc://@{server}/{database}?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes"

engine = create_engine(connection_string)

### Очистка таблиц  - если нужно импортировать все данные заново

In [ ]:
from sqlalchemy import text

# Список таблиц в правильном порядке (от дочерних к родительским)
tables_to_clear = [
    'OrderItem',
    'Order',
    'StockItem',
    'Product',
    'User',
    'Size',
    'Manufacturer',
    'Subcategory',
    'Category',
    'Role'
]

def clear_all_tables(engine):
    # Используем транзакцию, чтобы если что-то пойдет не так, всё откатилось
    with engine.begin() as conn:
        for table in tables_to_clear:
            try:
                # 1. Очищаем данные (безопаснее, чем TRUNCATE, если есть связи)
                conn.execute(text(f"DELETE FROM [{table}]"))
                # 2. Сбрасываем счетчик IDENTITY на начальное значение (1)
                conn.execute(text(f"DBCC CHECKIDENT ('{table}', RESEED, 0)"))
                print(f"Таблица {table} очищена и счетчик сброшен.")
            except Exception as e:
                print(f"Ошибка при очистке {table}: {e}")
                # Если произошла ошибка (например, из-за связей), транзакция откатится
                raise e 
    print("Готово")

# Вызываем функцию
clear_all_tables(engine)

### Загрузка всех таблиц с данными и подготовка для импорта в БД

In [ ]:
# таблицы с данными
orders = pd.read_excel('data/Orders_import.xlsx')
products= pd.read_excel('data/Products_import.xlsx')
sizes = pd.read_excel('data/Sizes_import.xlsx')
stock_items = pd.read_excel('data/Stock_items_import.xlsx')
users = pd.read_excel('data/Users_import.xlsx')

#### Таблицы-справочники: простой импорт без поиска соответствий ID

In [ ]:
# Импорт данных в БД
# таблицы-справочники
# уникальные роли из таблички
unique_roles = pd.DataFrame({'Name':users['Роль'].unique()})
# Добавим данные о ролях пользователя в базу данных
unique_roles.to_sql(name='Role', con=engine, if_exists='append', index=False)
# уникальные категории обуви из таблички
unique_categories = pd.DataFrame({'Name': products['Категория'].unique()})
unique_categories.to_sql(name = 'Category', con = engine,  if_exists='append', index=False)
# уникальные подкатегории
unique_subcategories = pd.DataFrame({'Name': products['Подкатегория'].unique()})
unique_subcategories.to_sql(name = 'Subcategory', con = engine, if_exists = 'append', index = False)
# уникальный производитель
unique_manufacturer = pd.DataFrame({'Name': products['Производство'].unique()})
unique_manufacturer.to_sql(name = 'Manufacturer', con = engine, if_exists='append', index = False)
# размеры
unique_sizes = pd.DataFrame({'Value': sizes['Размер'].dropna().unique()})
unique_sizes.to_sql(name = 'Size', con=engine, if_exists='append', index = False)

#### Таблица "Пользователи" - User

In [ ]:
# вместо Роли нужно вставить соответствующий Id из справочника
role_from_database = pd.DataFrame(pd.read_sql('SELECT * FROM Role', con = engine).rename(columns={'Name':'Роль'}))
# переименуем столбцы в соответствии с БД
users_for_db = (
    users
    .merge(role_from_database, how='left', on = 'Роль')
    .rename(columns = {
        'Фамилия':'Surname',
        'Имя':'Name',
        'Отчетсво':'Patronymic',
        'Логин':'Login',
        'Id':'RoleId'
    })
    .drop(columns = 'Роль')
)

users_for_db.to_sql(name = 'User', con = engine, if_exists='append', index = False)


#### Таблица "Товары" - Products

In [ ]:
products_to_db = (
    products
    .merge(pd.read_sql('SELECT * FROM Category', con = engine).rename(columns = {'Name':'Категория', 'Id':'CategoryId'}), how = 'left', on = 'Категория') # Вместо названий категории будет ее Id, соответствующий БД
    .merge(pd.read_sql('SELECT * FROM Subcategory', con = engine).rename(columns = {'Name':'Подкатегория', 'Id':'SubcategoryId'}), how = 'left', on = 'Подкатегория')
    .merge(pd.read_sql("SELECT * FROM  Manufacturer", con = engine).rename(columns = {'Name':'Производство', 'Id':'ManufacturerId'}), how = 'left', on = 'Производство')
    .drop(columns = ['Категория', 'Подкатегория', 'Производство'])
    .rename(columns = {'Изображение':'Photo',
                       'Наименование товара':'Name',
                       'Описание':'Description',
                       'Состав':'Composition',
                       'Цена':'Price'})
    [['Name', 'CategoryId', 'SubcategoryId', 'ManufacturerId','Description', 'Composition', 'Price', 'Photo']]
  
)

In [ ]:
products_to_db.to_sql(name = 'Product', con = engine, if_exists = 'append', index=False)

#### Таблица "Количество на складе" - StockItem

In [ ]:
size_table = pd.read_sql("SELECT * FROM Size", con = engine).rename(columns = {'Id':'SizeId', 'Value':'Размер'})
size_table['Размер'] = size_table['Размер'].astype('float64')

In [ ]:
stock_items_to_db = (
    stock_items
    .drop(columns = ['Производство'])
    .merge(pd.read_sql("SELECT id, Name FROM Product", con = engine).rename(columns = {'id':'ProductId', 'Name':'Наименование товара'}), how = 'left', on = 'Наименование товара')
    .merge(size_table, how = 'left', on = 'Размер')
    .drop(columns = ['Наименование товара', 'Размер'])
    .rename(columns = {'Количество доступное для заказа':'Quantity'})
    [['ProductId', 'SizeId', 'Quantity']]
    .dropna()  
)
stock_items_to_db['ProductId'] = stock_items_to_db['ProductId'].astype('int')

In [ ]:
stock_items_to_db.to_sql(name = 'StockItem', con = engine, if_exists = 'append', index = False)

#### Таблица "Заказы" - Order

In [ ]:
# Здесь только дата, Id заказа и соответствие пользователя заказу - то есть один к одному.
# Товары в заказе хранятся в отдельной таблице
orders_and_user = orders[['Номер заказа', 'Дата заказа', 'ФИО']].copy()
# В таблице Excel ФИО находятся в одном столбце. Их нужно разбить на отдельные, чтобы потом найти конкретный ID
orders_and_user['ФИО'] = orders_and_user['ФИО'].str.split(' ')
orders_and_user['Surname'] = orders_and_user['ФИО'].str[0]
orders_and_user['Name'] = orders_and_user['ФИО'].str[1]
orders_and_user['Patronymic'] = orders_and_user['ФИО'].str[2]
orders_and_user.drop(columns = 'ФИО', inplace = True)

In [ ]:
orders_and_user_for_db = (
    orders_and_user
    .merge(pd.read_sql("SELECT * FROM [User]", con = engine), how = 'left', on = ['Surname', 'Name', 'Patronymic'])
    [['Номер заказа', 'Дата заказа', 'Id']]
    .rename(columns = {'Номер заказа':'Id', 'Дата заказа':'OrderDate', 'Id':'UserId'})
    .drop_duplicates(['Id', 'OrderDate'])
    [['OrderDate', 'UserId']]
)
orders_and_user_for_db

In [ ]:
orders_and_user_for_db.to_sql(name = 'Order', con = engine, if_exists = 'append', index = False)

#### Таблица состав заказа OrderItem

In [ ]:
pd.read_sql('SELECT * FROM OrderItem', con = engine)

In [ ]:
orders_to_db = (
    orders[['Номер заказа', 'Наименование товара', 'Размер', 'Количество', 'Цена за единицу']]
    .rename(columns = {'Номер заказа':'OrderId',
                           'Наименование товара':'Name',
                           'Количество':'Quantity',
                           'Цена за единицу':'Price'})
        .merge(pd.read_sql("SELECT Id, Name FROM Product", con = engine), how = 'left', on = 'Name')
        .merge(size_table, how = 'left', on = 'Размер')
        .rename(columns = {'Id':'ProductId'})
        .merge(pd.read_sql('SELECT Id, ProductId, SizeId FROM StockItem', con = engine).rename(columns = {'Id':'StockItemId'}), how = 'left', on = ['ProductId', 'SizeId'])
        [['OrderId', 'StockItemId', 'Quantity', 'Price']]
        .dropna()
        
    )
orders_to_db['StockItemId'] = orders_to_db['StockItemId'].astype(int)

In [ ]:
orders_to_db.to_sql(name = 'OrderItem', con = engine, if_exists = 'append', index = False)